# Audio Scene-to-Image Generator

**Models:** CLAP + Stable Diffusion 1.5 + CLIP  
**Interface:** Gradio  
**Environment:** Google Colab with GPU (Runtime → Change runtime type → T4 GPU)

## Project workflow
1. Upload or record an environmental audio clip.
2. CLAP compares the audio with candidate sound descriptions and ranks likely categories.
3. The highest-ranked category selects a visual prompt.
4. Stable Diffusion generates an image from that prompt.
5. CLIP compares the generated image with candidate scene descriptions.
6. Gradio displays the generated image and relative scores.

**Important:** This notebook uses pretrained models and does not train them. It can choose only among the sound categories defined below. CLAP/CLIP scores are relative to these candidates, not calibrated probabilities. The generated image is an AI interpretation, not proof of the real-world scene. The first run downloads model weights and requires internet access.

## 1. Install dependencies

## 2. Imports and GPU check

## 3. Load the three pretrained models

Stable Diffusion runs on the GPU in half precision. CLAP and CLIP run on the CPU to conserve GPU memory.

## 4. Define the supported sound categories

## 5. Main pipeline: audio → scene prompt → generated image → CLIP evaluation

## 6. Launch the Gradio web interface

In [3]:

# ===============================================================
# AUDIO SCENE-TO-IMAGE GENERATOR
# Single-cell Google Colab application
# CLAP: audio understanding | Stable Diffusion: image generation
# CLIP: generated-image/text similarity | Gradio: user interface
# ===============================================================

import os
import gc
import numpy as np
import pandas as pd
import torch
import librosa
import gradio as gr

from transformers import (
    ClapModel,
    ClapProcessor,
    CLIPModel,
    CLIPProcessor
)
from diffusers import StableDiffusionPipeline

# ----------------------- CONFIGURATION --------------------------

AUDIO_MODEL_ID = "laion/clap-htsat-unfused"
IMAGE_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

if DEVICE == "cpu":
    print("WARNING: No GPU detected. Image generation will be slow.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

# ----------------------- SCENE DATABASE -------------------------

SCENES = {
    "Rain": {
        "audio": "The sound of rain falling steadily outdoors.",
        "visual": "rain falling on a city street"
    },
    "Thunderstorm": {
        "audio": "The sound of thunder rumbling during a storm.",
        "visual": "a dark stormy sky with lightning and heavy rain"
    },
    "Ocean Waves": {
        "audio": "The sound of ocean waves crashing onto a beach.",
        "visual": "ocean waves crashing onto a sandy beach"
    },
    "Birds": {
        "audio": "The sound of birds chirping and singing outdoors.",
        "visual": "birds perched on branches in a green forest"
    },
    "Forest": {
        "audio": "The ambient sounds of a forest and wildlife.",
        "visual": "a peaceful dense green forest with sunbeams"
    },
    "Traffic": {
        "audio": "The sound of vehicles driving along a busy road.",
        "visual": "cars and buses on a busy urban road"
    },
    "Train": {
        "audio": "The sound of a train moving on railway tracks.",
        "visual": "a passenger train moving along railway tracks"
    },
    "Aircraft": {
        "audio": "The sound of an airplane engine flying overhead.",
        "visual": "a passenger airplane flying through the clouds"
    },
    "Dog": {
        "audio": "The sound of a dog barking.",
        "visual": "a dog standing in a garden"
    },
    "Cat": {
        "audio": "The sound of a cat meowing.",
        "visual": "a domestic cat sitting beside a window"
    },
    "Fire": {
        "audio": "The sound of a fire crackling and burning.",
        "visual": "orange flames and glowing embers in a campfire"
    },
    "Wind": {
        "audio": "The sound of strong wind blowing through trees.",
        "visual": "trees bending in the wind beneath cloudy skies"
    },
    "People Talking": {
        "audio": "The sound of people talking and chatting together.",
        "visual": "people chatting together in an outdoor public place"
    },
    "Applause": {
        "audio": "The sound of a crowd clapping and applauding.",
        "visual": "an audience applauding at a live event"
    },
    "Footsteps": {
        "audio": "The sound of footsteps walking on a hard surface.",
        "visual": "a person walking along a paved pathway"
    },
    "Cooking": {
        "audio": "The sound of food sizzling and frying in a pan.",
        "visual": "food sizzling in a frying pan in a kitchen"
    },
    "Machinery": {
        "audio": "The sound of machinery operating inside a factory.",
        "visual": "industrial machinery operating inside a factory"
    },
    "Music": {
        "audio": "The sound of instrumental music being played.",
        "visual": "musical instruments on a stage under concert lights"
    }
}

SCENE_NAMES = list(SCENES.keys())
AUDIO_DESCRIPTIONS = [SCENES[s]["audio"] for s in SCENE_NAMES]
VISUAL_DESCRIPTIONS = [
    f"A realistic photograph of {SCENES[s]['visual']}."
    for s in SCENE_NAMES
]

# ----------------------- LOAD CLAP -------------------------------

print("\nLoading CLAP audio model...")
clap_processor = ClapProcessor.from_pretrained(AUDIO_MODEL_ID)

# Important: Keep CLAP in FP32 to avoid the reported cudnn_batch_norm
# error caused by FP32 inputs being sent to FP16 model weights.
clap_model = ClapModel.from_pretrained(
    AUDIO_MODEL_ID,
    torch_dtype=torch.float32
).to(DEVICE)

clap_model.eval()
print("CLAP loaded in FP32.")

# ----------------------- LOAD CLIP -------------------------------

print("\nLoading CLIP image-text model...")
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)

clip_model = CLIPModel.from_pretrained(
    CLIP_MODEL_ID,
    torch_dtype=torch.float32
).to(DEVICE)

clip_model.eval()
print("CLIP loaded in FP32.")

# ----------------------- LOAD STABLE DIFFUSION -------------------

print("\nLoading Stable Diffusion. First download may take several minutes...")

image_pipe = StableDiffusionPipeline.from_pretrained(
    IMAGE_MODEL_ID,
    torch_dtype=IMAGE_DTYPE,
    use_safetensors=True
).to(DEVICE)

image_pipe.set_progress_bar_config(disable=True)

# Helps reduce peak memory use on supported devices.
image_pipe.enable_attention_slicing()

print("Stable Diffusion loaded using", IMAGE_DTYPE)

# ----------------------- CLASSIFY AUDIO --------------------------

def classify_audio(audio_path):
    if not audio_path or not os.path.isfile(audio_path):
        raise ValueError("Upload or record an audio file first.")

    # Load up to 30 seconds of mono audio at CLAP's 48 kHz sample rate.
    waveform, _ = librosa.load(
        audio_path,
        sr=48000,
        mono=True,
        duration=30
    )

    if waveform.size == 0:
        raise ValueError("The audio file is empty or could not be decoded.")

    if float(np.max(np.abs(waveform))) < 1e-5:
        raise ValueError("The audio contains only silence.")

    audio_batch = clap_processor(
        audios=waveform,
        sampling_rate=48000,
        return_tensors="pt"
    )

    text_batch = clap_processor(
        text=AUDIO_DESCRIPTIONS,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # Preserve integer token inputs; explicitly keep audio features FP32.
    audio_inputs = {
        key: value.to(DEVICE, dtype=torch.float32)
        if value.is_floating_point()
        else value.to(DEVICE)
        for key, value in audio_batch.items()
        if isinstance(value, torch.Tensor)
    }

    text_inputs = {
        key: value.to(DEVICE)
        for key, value in text_batch.items()
        if key in ("input_ids", "attention_mask", "position_ids")
        and isinstance(value, torch.Tensor)
    }

    with torch.inference_mode():
        audio_features = clap_model.get_audio_features(**audio_inputs)
        text_features = clap_model.get_text_features(**text_inputs)

        audio_features = torch.nn.functional.normalize(
            audio_features.float(), p=2, dim=-1
        )
        text_features = torch.nn.functional.normalize(
            text_features.float(), p=2, dim=-1
        )

        similarities = (audio_features @ text_features.T).squeeze(0)
        scores = torch.softmax(similarities * 10.0, dim=-1)

    scores = scores.detach().cpu().numpy()
    ranked = np.argsort(scores)[::-1]
    best_scene = SCENE_NAMES[int(ranked[0])]

    table = pd.DataFrame({
        "Rank": list(range(1, min(6, len(ranked) + 1))),
        "Predicted scene": [SCENE_NAMES[int(i)] for i in ranked[:5]],
        "Relative score (%)": [
            round(float(scores[i]) * 100, 2) for i in ranked[:5]
        ]
    })

    return best_scene, table

# ----------------------- CLIP IMAGE EVALUATION -------------------

def evaluate_generated_image(image):
    image_batch = clip_processor(
        images=image,
        return_tensors="pt"
    )

    text_batch = clip_processor(
        text=VISUAL_DESCRIPTIONS,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    pixel_values = image_batch["pixel_values"].to(
        DEVICE, dtype=torch.float32
    )

    text_inputs = {
        key: value.to(DEVICE)
        for key, value in text_batch.items()
        if key in ("input_ids", "attention_mask")
    }

    with torch.inference_mode():
        image_features = clip_model.get_image_features(
            pixel_values=pixel_values
        )
        text_features = clip_model.get_text_features(**text_inputs)

        image_features = torch.nn.functional.normalize(
            image_features.float(), p=2, dim=-1
        )
        text_features = torch.nn.functional.normalize(
            text_features.float(), p=2, dim=-1
        )

        similarities = (
            image_features @ text_features.T
        ).squeeze(0)

        relative_scores = torch.softmax(similarities * 10.0, dim=-1)

    relative_scores = relative_scores.cpu().numpy()
    ranked = np.argsort(relative_scores)[::-1]

    table = pd.DataFrame({
        "Rank": list(range(1, min(6, len(ranked) + 1))),
        "Matching visual scene": [
            SCENE_NAMES[int(i)] for i in ranked[:5]
        ],
        "Relative score (%)": [
            round(float(relative_scores[i]) * 100, 2)
            for i in ranked[:5]
        ]
    })

    return table

# ----------------------- GENERATE IMAGE --------------------------

def generate_scene(audio_path, custom_prompt, steps, seed):
    if not audio_path:
        raise gr.Error("Upload or record an audio file first.")

    try:
        # 1. Classify the audio.
        scene, audio_results = classify_audio(audio_path)

        # 2. Create a prompt from the detected category or user's prompt.
        if custom_prompt and custom_prompt.strip():
            prompt = custom_prompt.strip()
        else:
            prompt = (
                f"A realistic photograph of {SCENES[scene]['visual']}, "
                "natural lighting, detailed environment, high quality"
            )

        negative_prompt = (
            "blurry, low quality, distorted, deformed, "
            "watermark, illegible text, oversaturated"
        )

        generator = torch.Generator(device=DEVICE).manual_seed(int(seed))

        # 3. Generate the image using Stable Diffusion.
        with torch.inference_mode():
            generated_image = image_pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=int(steps),
                guidance_scale=7.0,
                height=512,
                width=512,
                generator=generator
            ).images[0]

        # 4. Compare the generated image with visual scene descriptions.
        clip_results = evaluate_generated_image(generated_image)

        # 5. Save generated image for download.
        output_path = "/content/generated_audio_scene.png"
        generated_image.save(output_path)

        explanation = (
            f"### Audio scene prediction: {scene}\n\n"
            f"**Generation prompt:** {prompt}\n\n"
            "CLAP scores rank audio-text matches. CLIP scores rank "
            "image-text matches. Both are relative model scores, not "
            "calibrated probabilities or guarantees of correctness."
        )

        return (
            generated_image,
            explanation,
            audio_results,
            clip_results,
            output_path
        )

    except gr.Error:
        raise
    except Exception as exc:
        raise gr.Error(
            f"{type(exc).__name__}: {exc}"
        ) from exc
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

# ----------------------- GRADIO UI -------------------------------

with gr.Blocks(title="Audio Scene-to-Image Generator") as demo:
    gr.Markdown(
        """
        # Audio Scene-to-Image Generator
        Upload an environmental sound or record audio using your
        microphone. The system classifies the sound and generates
        a related visual scene.

        **Pipeline:** Audio → CLAP → Scene prompt → Stable Diffusion
        → Generated image → CLIP image-text evaluation
        """
    )

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="Upload or record audio"
            )

            prompt_input = gr.Textbox(
                label="Optional custom image prompt",
                placeholder="Leave blank to generate from detected audio."
            )

            steps_input = gr.Slider(
                minimum=10,
                maximum=35,
                value=20,
                step=5,
                label="Generation steps"
            )

            seed_input = gr.Number(
                value=42,
                precision=0,
                label="Random seed"
            )

            generate_button = gr.Button(
                "Analyze Audio and Generate Image",
                variant="primary"
            )

        with gr.Column():
            image_output = gr.Image(
                label="Generated image",
                type="pil"
            )
            explanation_output = gr.Markdown()
            download_output = gr.File(
                label="Download generated image"
            )

    audio_results_output = gr.Dataframe(
        label="Top audio scene matches (CLAP)",
        interactive=False
    )

    clip_results_output = gr.Dataframe(
        label="Top visual scene matches (CLIP)",
        interactive=False
    )

    generate_button.click(
        fn=generate_scene,
        inputs=[
            audio_input,
            prompt_input,
            steps_input,
            seed_input
        ],
        outputs=[
            image_output,
            explanation_output,
            audio_results_output,
            clip_results_output,
            download_output
        ]
    )

print("\nApplication ready. Opening Gradio...")
demo.queue().launch(share=True, debug=True)


GPU: Tesla T4

Loading CLAP audio model...
CLAP loaded in FP32.

Loading CLIP image-text model...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (7 outputs left out)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Stable Diffusion loaded using torch.float16

Application ready. Opening Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e7f5990fc8ab1b3877.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e7f5990fc8ab1b3877.gradio.live


In [4]:

# ================================================================
# IMAGE-TO-DESCRIPTION-TO-SPEECH GENERATOR
# Google Colab | Hugging Face BLIP | gTTS | Gradio
# Single-cell implementation
# ================================================================

# 1. INSTALL MISSING DEPENDENCIES
import sys
import os
import gc
import tempfile
import subprocess
import importlib.util

packages = {
    "gradio": "gradio",
    "transformers": "transformers",
    "PIL": "pillow",
    "gtts": "gTTS"
}

missing = [
    package for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

# 2. IMPORT LIBRARIES
import torch
import gradio as gr

from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from gtts import gTTS

# Release models from the previous audio-to-image application,
# if those objects are still present in this notebook session.
for name in [
    "image_pipe", "clap_model", "clap_processor",
    "clip_model", "clip_processor"
]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 3. CONFIGURATION
MODEL_ID = "Salesforce/blip-image-captioning-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Loading image captioning model. First run downloads model files...")

# Keep BLIP in FP32 for compatibility.
processor = BlipProcessor.from_pretrained(MODEL_ID)
model = BlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
).to(DEVICE)

model.eval()
print("BLIP model loaded successfully.")

# 4. IMAGE UNDERSTANDING AND DESCRIPTION
def describe_image(image):
    if image is None:
        raise gr.Error("Please upload an image first.")

    try:
        image = image.convert("RGB")

        # Preprocess the image.
        inputs = processor(
            images=image,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        # Generate a natural-language image caption.
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=60,
                num_beams=5,
                repetition_penalty=1.2
            )

        description = processor.decode(
            output_ids[0],
            skip_special_tokens=True
        ).strip()

        if not description:
            raise ValueError("The model did not generate a description.")

        # Normalize capitalization and punctuation.
        description = description[0].upper() + description[1:]
        if description[-1] not in ".!?":
            description += "."

        return description

    except gr.Error:
        raise
    except Exception as error:
        raise gr.Error(
            f"Image description failed: {type(error).__name__}: {error}"
        ) from error


# 5. TEXT-TO-SPEECH
def create_speech(image, edited_description):
    if image is None:
        raise gr.Error("Please upload an image first.")

    if not edited_description or not edited_description.strip():
        raise gr.Error("Please generate or enter a description first.")

    description = edited_description.strip()

    try:
        # gTTS requires internet access to generate speech.
        with tempfile.NamedTemporaryFile(
            suffix=".mp3",
            delete=False
        ) as temp_audio:
            audio_path = temp_audio.name

        tts = gTTS(
            text=description,
            lang="en",
            slow=False
        )
        tts.save(audio_path)

        if not os.path.exists(audio_path) or os.path.getsize(audio_path) == 0:
            raise RuntimeError("The audio file was not created.")

        return audio_path, audio_path

    except Exception as error:
        if "audio_path" in locals() and os.path.exists(audio_path):
            os.remove(audio_path)

        raise gr.Error(
            "Text-to-speech failed. Check your internet connection "
            f"and try again. Details: {type(error).__name__}: {error}"
        ) from error


# 6. COMBINED PROCESSING
def process_image(image):
    description = describe_image(image)
    audio_path, download_path = create_speech(image, description)

    return description, audio_path, download_path


# 7. GRADIO APPLICATION
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Image Description and Speech Generator"
) as demo:

    gr.Markdown(
        """
        # Image-to-Description-to-Speech Generator

        Upload a photograph. The AI will understand its visual content,
        generate a natural-language description, and read that description
        aloud.

        **Pipeline:** Image → BLIP image captioning → Text description
        → Text-to-speech → MP3 audio
        """
    )

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload an image"
            )

            generate_button = gr.Button(
                "Describe Image and Generate Speech",
                variant="primary"
            )

            gr.Markdown(
                "Supported formats include common image files such as "
                "JPG, JPEG, and PNG."
            )

        with gr.Column():
            description_output = gr.Textbox(
                label="Generated description (editable)",
                lines=5,
                placeholder="The image description will appear here."
            )

            audio_output = gr.Audio(
                label="Spoken description",
                type="filepath",
                autoplay=False
            )

            download_output = gr.File(
                label="Download speech as MP3"
            )

    # One-click image description and speech generation.
    generate_button.click(
        fn=process_image,
        inputs=[image_input],
        outputs=[
            description_output,
            audio_output,
            download_output
        ]
    )

    # Regenerate speech after the user edits the description.
    speak_button = gr.Button("Generate Speech from Edited Text")
    speak_button.click(
        fn=create_speech,
        inputs=[image_input, description_output],
        outputs=[audio_output, download_output]
    )

    gr.Markdown(
        """
        ### Example
        Upload a dog playing in a park. The application may describe
        a dog on grass in an outdoor setting, then read the generated
        description aloud.

        Note: The caption is generated by a pretrained model and may
        not identify every object or action in an image.
        """
    )

print("Launching application...")
demo.queue().launch(share=True, debug=True)


Installing missing packages: ['gTTS']
Device: cuda
Loading image captioning model. First run downloads model files...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (3 outputs left out)

/tmp/ipykernel_3935/894909515.py:171: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://68da33fafcb136e96b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://68da33fafcb136e96b.gradio.live


## 7. Testing and troubleshooting

Test with clear, short recordings such as rain, birds, ocean waves, traffic, a passing train, or a dog barking.

- **GPU error:** select a GPU runtime in Colab, then rerun the notebook from the start.
- **Out of memory:** lower inference steps to 15; restart the runtime if required.
- **Unexpected category:** this notebook chooses among only the 18 categories above. Add more candidate descriptions to `SCENES` for broader coverage.
- **Slow first run:** model weights must be downloaded. Keep the runtime active while testing.
- **Score interpretation:** displayed percentages are softmax-normalized rankings among candidate labels. They are not calibrated probabilities or objective image-quality scores.

## Architecture to explain in your presentation

**Audio waveform → CLAP audio–text matching → selected scene prompt → Stable Diffusion denoising → generated image → CLIP image–text comparison → Gradio results.**